In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Notebook 14 — Cross-Dataset Comprehensive Evaluation

Zero-shot transfer: train on dataset A, evaluate on dataset B.  
Covers all 6 cross-dataset pairs × 2 techniques (BanglaBERT baseline + FGM ε=0.5).  
GPU required. Resume-safe.

In [ ]:
import os, json, time, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score, classification_report
warnings.filterwarnings("ignore")

IS_KAGGLE = os.path.exists("/kaggle/working")
REPO = Path("/kaggle/working/Sarcasm_detection") if IS_KAGGLE else Path.cwd().parent

if IS_KAGGLE:
    HF_CACHE = Path("/kaggle/tmp/hf_cache")
    CKPT_DIR = Path("/kaggle/tmp/checkpoints")
    HF_CACHE.mkdir(parents=True, exist_ok=True)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"]            = str(HF_CACHE)
    os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE)
else:
    CKPT_DIR = Path("/tmp/nb14_checkpoints")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS_DIR  = REPO / "01_data" / "interim" / "splits"
TABLES_DIR  = REPO / "04_outputs" / "tables"
PREDS_DIR   = REPO / "04_outputs" / "predictions"
REPORTS_DIR = REPO / "04_outputs" / "reports"
LOGS_DIR    = REPO / "04_outputs" / "run_logs"
for d in [TABLES_DIR, PREDS_DIR, REPORTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = (DEVICE == "cuda")
RESULTS_CSV = TABLES_DIR / "14_cross_dataset_results.csv"

print(f"ENV    = {'kaggle' if IS_KAGGLE else 'local'}")
print(f"DEVICE = {DEVICE}  (FP16={USE_FP16})")
print(f"REPO   = {REPO}")


## Dataset utilities

In [ ]:
BACKBONE_HF = "csebuetnlp/banglabert"
TOKENIZER   = None   # lazy-loaded once
MAX_LEN     = 128
LABEL_COL   = "label_binary"
NUM_LABELS  = 2

DATASET_DISPLAY = {
    "banglasarc":  "BanglaSarc",
    "banglasarc3": "BanglaSarc3",
    "ben_sarc":    "Ben-Sarc",
}

def get_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        TOKENIZER = AutoTokenizer.from_pretrained(BACKBONE_HF)
    return TOKENIZER

def load_split(dataset_name, split, task="binary"):
    """Load a split CSV and return a tokenised HF Dataset."""
    path = SPLITS_DIR / f"{dataset_name}_{task}_{split}.csv"
    df   = pd.read_csv(path)[["text", LABEL_COL]].rename(columns={LABEL_COL: "labels"})
    df["labels"] = df["labels"].astype(int)
    tok = get_tokenizer()
    def tokenize(batch):
        return tok(batch["text"], truncation=True,
                   padding="max_length", max_length=MAX_LEN)
    ds = Dataset.from_pandas(df, preserve_index=False)
    ds = ds.map(tokenize, batched=True).remove_columns(["text"])
    ds.set_format("torch")
    return ds

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "macro_f1":    f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
        "accuracy":    accuracy_score(labels, preds),
    }

print("Dataset utilities ready.")


## Trainer classes

In [ ]:
class _AdvTrainerBase(Trainer):
    """Provides _safe_backward() compatible with local CPU and Kaggle FP16."""
    def _safe_backward(self, loss):
        try:
            self.accelerator.backward(loss)          # new transformers + accelerate
        except AttributeError:
            if hasattr(self, "scaler") and self.scaler is not None:
                self.scaler.scale(loss).backward()   # manual FP16 scaler
            else:
                loss.backward()                      # local CPU


class FGMTrainer(_AdvTrainerBase):
    """Fast Gradient Method adversarial training."""
    def __init__(self, *args, fgm_eps=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.fgm_eps = fgm_eps

    def _fgm_attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None and "embedding" in name:
                norm = param.grad.norm()
                if norm != 0:
                    param.data.add_(self.fgm_eps * param.grad / norm)

    def _fgm_restore(self, backup):
        for name, param in self.model.named_parameters():
            if name in backup:
                param.data = backup[name]

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels  = inputs.get("labels")   # get not pop — pop breaks adversarial second pass
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        loss    = nn.CrossEntropyLoss()(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        # ── clean forward + backward ──────────────────────────────────────────
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        loss_to_return = loss.detach().clone()   # capture BEFORE backward
        self._safe_backward(loss)

        # ── FGM: perturb embeddings ───────────────────────────────────────────
        backup = {n: p.data.clone()
                  for n, p in model.named_parameters()
                  if p.requires_grad and p.grad is not None and "embedding" in n}
        self._fgm_attack()

        # ── adversarial forward + backward ────────────────────────────────────
        with self.compute_loss_context_manager():
            loss_adv = self.compute_loss(model, inputs)
        self._safe_backward(loss_adv)
        self._fgm_restore(backup)

        return loss_to_return

print("Trainer classes ready.")


## Run matrix

In [ ]:
TECHNIQUES = {
    "bb_baseline": {"trainer_cls": Trainer,     "fgm_eps": None},
    "bb_fgm_e05":  {"trainer_cls": FGMTrainer,  "fgm_eps": 0.5},
}
DATASETS_BINARY = ["banglasarc", "banglasarc3", "ben_sarc"]

# All cross-dataset pairs (A → B, A ≠ B)
RUN_MATRIX = []
for technique in TECHNIQUES:
    for train_ds in DATASETS_BINARY:
        for test_ds in DATASETS_BINARY:
            if train_ds == test_ds:
                continue
            RUN_MATRIX.append({
                "run_id":    f"{technique}_{train_ds}__to__{test_ds}",
                "technique": technique,
                "train_ds":  train_ds,
                "test_ds":   test_ds,
            })

# Order: short train datasets first to maximise completed runs before timeout
TRAIN_ORDER = {"banglasarc": 0, "banglasarc3": 1, "ben_sarc": 2}
RUN_MATRIX.sort(key=lambda r: (TRAIN_ORDER[r["train_ds"]], r["technique"], r["test_ds"]))

print(f"Total cross-dataset runs: {len(RUN_MATRIX)}")
for r in RUN_MATRIX:
    print(f"  [{r['technique']:12s}]  {r['train_ds']:12s}  →  {r['test_ds']}")


## Training + transfer function

In [ ]:
SCHEMA = ["run_id", "technique", "backbone", "train_dataset", "test_dataset",
          "task", "test_macro_f1", "test_weighted_f1", "test_accuracy",
          "val_macro_f1_on_train_ds", "time_seconds"]


def train_and_transfer(run_id, technique, train_ds, test_ds,
                       task="binary"):
    """Fine-tune BanglaBERT on train_ds; evaluate zero-shot on test_ds."""
    cfg = TECHNIQUES[technique]
    t0  = time.time()

    print(f"\n{'='*70}")
    print(f"  RUN : {run_id}")
    print(f"  tech={technique}  train={train_ds}  →  test_ds={test_ds}")
    print(f"{'='*70}")

    # ── data ─────────────────────────────────────────────────────────────────
    train_data = load_split(train_ds, "train", task)
    val_data   = load_split(train_ds, "val",   task)   # val on SOURCE domain
    test_data  = load_split(test_ds,  "test",  task)   # zero-shot on TARGET domain
    print(f"  train={len(train_data)}  val={len(val_data)}  target_test={len(test_data)}")

    # ── training schedule ─────────────────────────────────────────────────────
    steps_per_epoch = max(1, len(train_data) // (4 * 2))  # batch=4, grad_acc=2
    total_steps     = steps_per_epoch * 3
    warmup_steps    = max(1, int(total_steps * 0.1))

    # ── model ─────────────────────────────────────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        BACKBONE_HF, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
    if IS_KAGGLE:
        model.gradient_checkpointing_enable()

    run_ckpt = CKPT_DIR / run_id
    run_ckpt.mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir                  = str(run_ckpt),
        num_train_epochs            = 3,
        per_device_train_batch_size = 4,
        per_device_eval_batch_size  = 16,
        gradient_accumulation_steps = 2,
        learning_rate               = 2e-5,
        weight_decay                = 0.01,
        warmup_steps                = warmup_steps,
        fp16                        = USE_FP16,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "macro_f1",
        greater_is_better           = True,
        save_total_limit            = 1,
        seed                        = 42,
        logging_steps               = 50,
        report_to                   = "none",
        dataloader_pin_memory       = IS_KAGGLE,
    )

    trainer_kwargs = dict(
        model           = model,
        args            = args,
        train_dataset   = train_data,
        eval_dataset    = val_data,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
    )
    if cfg["fgm_eps"] is not None:
        trainer_kwargs["fgm_eps"] = cfg["fgm_eps"]

    trainer = cfg["trainer_cls"](**trainer_kwargs)
    trainer.train()

    # ── val score on source domain ────────────────────────────────────────────
    val_mf1 = trainer.evaluate(val_data).get("eval_macro_f1", float("nan"))

    # ── zero-shot eval on target domain ──────────────────────────────────────
    test_out  = trainer.evaluate(test_data)
    test_mf1  = test_out.get("eval_macro_f1",    float("nan"))
    test_wf1  = test_out.get("eval_weighted_f1", float("nan"))
    test_acc  = test_out.get("eval_accuracy",    float("nan"))

    # ── save predictions ──────────────────────────────────────────────────────
    pred_out  = trainer.predict(test_data)
    preds_arr = pred_out.predictions.argmax(axis=-1)
    probs_arr = torch.softmax(torch.tensor(pred_out.predictions.astype(float)), dim=-1).numpy()
    pred_df   = pd.DataFrame({
        "true_label": pred_out.label_ids,
        "pred_label": preds_arr,
        "prob_0":     probs_arr[:, 0],
        "prob_1":     probs_arr[:, 1],
    })
    pred_df.to_csv(PREDS_DIR / f"14_{run_id}_test_preds.csv", index=False)

    # ── classification report ─────────────────────────────────────────────────
    report = classification_report(pred_out.label_ids, preds_arr, digits=4)
    print(report)
    (REPORTS_DIR / f"14_{run_id}_report.txt").write_text(report)

    # ── cleanup ───────────────────────────────────────────────────────────────
    if run_ckpt.exists():
        shutil.rmtree(run_ckpt)
    del model, trainer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    elapsed = round(time.time() - t0, 1)
    print(f"  → test_macro_f1={test_mf1:.4f}  val_mf1_src={val_mf1:.4f}  time={elapsed}s")

    row = dict(run_id=run_id, technique=technique, backbone=BACKBONE_HF,
               train_dataset=train_ds, test_dataset=test_ds, task=task,
               test_macro_f1=round(test_mf1,4), test_weighted_f1=round(test_wf1,4),
               test_accuracy=round(test_acc,4), val_macro_f1_on_train_ds=round(val_mf1,4),
               time_seconds=elapsed)
    (LOGS_DIR / f"14_{run_id}_log.json").write_text(json.dumps(row, indent=2))
    return row

print("Training function ready.")


## Main loop (resume-safe)

In [ ]:
if RESULTS_CSV.exists():
    res_df = pd.read_csv(RESULTS_CSV)
    done   = set(res_df["run_id"])
    print(f"Resuming: {len(done)} run(s) already in {RESULTS_CSV}")
else:
    res_df = pd.DataFrame(columns=SCHEMA)
    done   = set()
    print(f"Starting fresh. {len(RUN_MATRIX)} runs to do.")

for run in RUN_MATRIX:
    run_id = run["run_id"]
    if run_id in done:
        print(f"  [SKIP] {run_id}")
        continue
    try:
        row    = train_and_transfer(**{k: v for k, v in run.items()})
        res_df = pd.concat([res_df, pd.DataFrame([row])], ignore_index=True)
        res_df.to_csv(RESULTS_CSV, index=False)   # incremental save
    except Exception as e:
        print(f"  [ERR ] {run_id}: {e}")
        import traceback; traceback.print_exc()

print(f"\nTotal runs in results: {len(res_df)}")


## Transfer matrices + degradation analysis

In [ ]:
# In-domain results (diagonal) from nb05/nb06
IN_DOMAIN = {
    ("bb_baseline", "banglasarc",  "banglasarc"):  0.9834,
    ("bb_baseline", "banglasarc3", "banglasarc3"): 0.7666,
    ("bb_baseline", "ben_sarc",    "ben_sarc"):    0.8018,
    ("bb_fgm_e05",  "banglasarc",  "banglasarc"):  0.9896,
    ("bb_fgm_e05",  "banglasarc3", "banglasarc3"): 0.7880,
    ("bb_fgm_e05",  "ben_sarc",    "ben_sarc"):    0.8022,
}

SEP = "=" * 72

for technique in TECHNIQUES:
    print(f"\n{SEP}")
    print(f"  TRANSFER MATRIX — {technique}  (Test Macro-F1)")
    print(f"  Rows = Train  |  Columns = Test  |  * = in-domain")
    print(SEP)
    sub = res_df[res_df.technique == technique]
    rows = []
    for tr in DATASETS_BINARY:
        row = {"Train \\ Test": DATASET_DISPLAY[tr]}
        for te in DATASETS_BINARY:
            if tr == te:
                v = IN_DOMAIN.get((technique, tr, tr), float("nan"))
                row[DATASET_DISPLAY[te]] = f"{v:.4f}*"
            else:
                m = sub[(sub.train_dataset == tr) & (sub.test_dataset == te)]
                row[DATASET_DISPLAY[te]] = f"{m.iloc[0].test_macro_f1:.4f}" if not m.empty else "—"
        rows.append(row)
    print(pd.DataFrame(rows).set_index("Train \\ Test").to_string())

# ── Delta table: FGM gain over baseline on transfer tasks ────────────────────
print(f"\n{SEP}")
print("  FGM vs BASELINE — Transfer Macro-F1 delta  (FGM − Baseline)")
print(SEP)

base_sub = res_df[res_df.technique == "bb_baseline"]
fgm_sub  = res_df[res_df.technique == "bb_fgm_e05"]

delta_rows = []
for _, brow in base_sub.iterrows():
    frow = fgm_sub[(fgm_sub.train_dataset == brow.train_dataset) &
                   (fgm_sub.test_dataset  == brow.test_dataset)]
    if frow.empty: continue
    delta = round(frow.iloc[0].test_macro_f1 - brow.test_macro_f1, 4)
    delta_rows.append({
        "train → test":     f"{brow.train_dataset} → {brow.test_dataset}",
        "baseline_f1":      brow.test_macro_f1,
        "fgm_f1":           frow.iloc[0].test_macro_f1,
        "delta (FGM-Base)": delta,
        "FGM wins": "YES" if delta > 0 else "NO",
    })

if delta_rows:
    print(pd.DataFrame(delta_rows).sort_values("delta (FGM-Base)",ascending=False).to_string(index=False))

# ── Degradation: in-domain vs transfer ───────────────────────────────────────
print(f"\n{SEP}")
print("  DEGRADATION — In-domain vs Transfer  (in_domain − transfer)")
print(SEP)

deg_rows = []
for _, row in res_df.iterrows():
    in_f1 = IN_DOMAIN.get((row.technique, row.train_dataset, row.train_dataset), float("nan"))
    deg_rows.append({
        "technique":    row.technique,
        "train→test":   f"{row.train_dataset}→{row.test_dataset}",
        "in_domain_f1": in_f1,
        "transfer_f1":  row.test_macro_f1,
        "degradation":  round(in_f1 - row.test_macro_f1, 4),
    })

if deg_rows:
    ddf = (pd.DataFrame(deg_rows)
             .sort_values(["technique","degradation"], ascending=[True,False]))
    print(ddf.to_string(index=False))

print(f"\n{'='*72}")
print("  NOTEBOOK 14 COMPLETE")
print(f"{'='*72}")
print("Outputs:")
print("  04_outputs/tables/14_cross_dataset_results.csv")
print("  04_outputs/tables/14_transfer_matrix_bb_baseline.csv")
print("  04_outputs/tables/14_transfer_matrix_bb_fgm_e05.csv")
print("  04_outputs/predictions/14_*_test_preds.csv")
print("  04_outputs/reports/14_*_report.txt")
print("  04_outputs/run_logs/14_*_log.json")
print("\nNext → Notebook 15: Domain Adaptation")


## Save outputs

In [ ]:
res_df.to_csv(RESULTS_CSV, index=False)

for technique in TECHNIQUES:
    sub = res_df[res_df.technique == technique]
    mat_rows = []
    for tr in DATASETS_BINARY:
        row = {"train_dataset": tr}
        for te in DATASETS_BINARY:
            if tr == te:
                row[te] = IN_DOMAIN.get((technique, tr, tr), float("nan"))
            else:
                m = sub[(sub.train_dataset == tr) & (sub.test_dataset == te)]
                row[te] = m.iloc[0].test_macro_f1 if not m.empty else float("nan")
        mat_rows.append(row)
    mat_path = TABLES_DIR / f"14_transfer_matrix_{technique}.csv"
    pd.DataFrame(mat_rows).set_index("train_dataset").to_csv(mat_path)

print("All artifacts saved.\n")
print("Files produced:")
for p in sorted(TABLES_DIR.glob("14_*.csv")):
    print(f"  {p.relative_to(REPO)}  ({p.stat().st_size/1024:.1f} KB)")
